# Processing the DC books to a df 

In [6]:
import re
import pandas as pd
from nltk.tokenize import sent_tokenize
import nltk
import os

In [7]:
books = {

    "AinUTL": {
        "file": "text_files/AinUTL.txt",
        "start_marker": "===== ch1.html =====",
        "end_marker": "===== cop.html =====",
        "chapter_pattern": r"===== ch(\d+)\.html =====",
        "date_pattern": r"""
            \b
            le\s+\d{1,2}\s+
            (janvier|février|fevrier|mars|avril|mai|juin|juillet|août|aout|
             septembre|octobre|novembre|décembre|decembre)
            \s+\d{4}
            \b
        """         # le 8 mai 1666
    }, 

    "AToBD": {
        "file": "text_files/AToBD.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """          # May 15, 1862
    },

    "BFfH": {
        "file": "text_files/BFfH.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """         # December 25, 1916
    },

    "BUoL": {
        "file": "text_files/BUoL.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            le\s+\d{1,2}\s+
            (janvier|février|fevrier|mars|avril|mai|juin|juillet|août|aout|
             septembre|octobre|novembre|décembre|decembre)
            \s+\d{4}
            \b
        """         # Le 31 décembre 1884
    },

    "CofOO": {
        "file": "text_files/CofOO.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b(
                # Full date with weekday (handles broken lines)
                (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{1,2}
                (?:\s*\n?\s*(?:st|nd|rd|th))?
                [\s,\n]*\d{4}

                |

                # Month + Year (January 1864)
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{4}

                |

                # Year only
                \b\d{4}\b

                # Sunday, April 29
                # th
                # , 1866
            )
        """
    },

    "DofmyC": {
        "file": "text_files/dofmyc.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b
            le\s+\d{1,2}\s+
            (janvier|février|fevrier|mars|avril|mai|juin|juillet|août|aout|
             septembre|octobre|novembre|décembre|decembre)
            \s+\d{4}
            \b
        """
    },

    "DofTT": {
        "file": "text_files/DofTT.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """         # May 19, 1887
    }, 

    
    "DRtoF": {
        "file": "text_files/drtof.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b(
                # Full date with weekday (handles broken lines)
                (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{1,2}
                (?:\s*\n?\s*(?:st|nd|rd|th))?
                [\s,\n]*\d{4}

                |

                # Month + Year (January 1864)
                (January|February|March|April|May|June|July|August|September|
                October|November|December)
                \s+\d{4}

                |

                # Year only
                \b\d{4}\b

                # Monday, January 12
                # th
                # , 1863
            )
        """
    },

    "EftW": {
        "file": "text_files/EftW.txt",
        "start_marker": "===== c02.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """
            # Monday, June 17, 1940
    },

    "FitS": {
        "file": "text_files/FitS.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}
            \b
        """
    },

    "NbutC": {
        "file": "text_files/nbutc.txt",
        "start_marker": "===== ch1.html =====",
        "end_marker": "===== cop.html =====",
        "chapter_pattern": r"===== ch(\d+)\.html =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}(?:st|nd|rd|th),\s*\d{4}
            \b
        """
    },

    "NSH": {
        "file": "text_files/NSH.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """
        # Wednesday, September 26, 1917
    },

    "OA": {
        "file": "text_files/OA.txt",
        "start_marker": "===== 9781443119849_epub_c01_r1.htm =====",
        "end_marker": "===== 9781443119849_epub_cop_r1.htm =====",
        "chapter_pattern": r"===== 9781443119849_epub_c(\d+)_r1.htm =====",
        "date_pattern":r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """ 
        # Sunday, November 12, 1922
    },

    "OatmyD": {
        "file": "text_files/OatmyD.txt",
        "start_marker": "===== ch01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== ch(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b

            # Monday, May 24, 1897, 10 in the morning
            # Tuesday, May 25, After supper
        """
    },

    "PasWide": {
        "file": "text_files/PasWide.txt",
        "start_marker": "===== 9781443113342_epub_c01.htm =====",
        "end_marker": "===== 9781443113342_epub_cop_r1.htm =====",
        "chapter_pattern": r"===== 9781443113342_epub_c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2}
            \b
        """
    },

    "PintPL": {
        "file": "text_files/PintPL.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """
    },
        
    "SofS": {
        "file": "text_files/SofS.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== (c\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
             October|November|December)
            \s+\d{1,2},\s*\d{4}
            \b
        """
    },

    "TA": {
        "file": "text_files/TA.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """
    },

    "TFN": {
        "file": "text_files/TFN.txt",
        "start_marker": "===== ch01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== ch(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{1,2}
            (?:,\s*\d{4})?
            \b
        """
    }, 

    "WofW": {
        "file": "text_files/WofW.txt",
        "start_marker": "===== c01.htm =====",
        "end_marker": "===== cop.htm =====",
        "chapter_pattern": r"===== c(\d+)\.htm =====",
        "date_pattern": r"""
            \b
            \d{1,2}\s+
            (January|February|March|April|May|June|July|August|September|
            October|November|December)
            \s+\d{4}
            \b
        """
    }
}

In [8]:
# Process each book in the dictionary
book_dfs = {}

for book_name, book_info in books.items():
    with open(book_info["file"], "r", encoding="utf-8") as f:
        raw_text = f.read()

    start = raw_text.find(book_info["start_marker"])
    end = raw_text.find(book_info["end_marker"])
    diary_text = raw_text[start:end]

    sections = re.split(book_info["chapter_pattern"], diary_text)

    chapters = []
    for i in range(1, len(sections), 2):
        chapter_id = sections[i]
        text = sections[i+1]

        chapters.append({
            "book": book_name,
            "chapter_id": chapter_id,
            "text": text.strip()
        })

    chapters_df = pd.DataFrame(chapters)
    book_dfs[book_name] = chapters_df
    print(f"Chapters for {book_name}:")
    print(chapters_df.head())

Chapters for AinUTL:
     book chapter_id                                               text
0  AinUTL          1  Dear Canada: Alone in an Untamed Land\n\n\n\n\...
1  AinUTL         10  Dear Canada: Alone in an Untamed Land\n\n\n\n\...
2  AinUTL         11  Dear Canada: Alone in an Untamed Land\n\n\n\n\...
3  AinUTL         12  Dear Canada: Alone in an Untamed Land\n\n\n\n\...
4  AinUTL         13  Dear Canada: Alone in an Untamed Land\n\n\n\n\...
Chapters for AToBD:
    book chapter_id                                               text
0  AToBD         01  Dear Canada: A Trail of Broken Dreams\n\n\n\n\...
1  AToBD         02  Dear Canada: A Trail of Broken Dreams\n\n\n\n\...
2  AToBD         03  Dear Canada: A Trail of Broken Dreams\n\n\n\n\...
3  AToBD         04  Dear Canada: A Trail of Broken Dreams\n\n\n\n\...
4  AToBD         05  Dear Canada: A Trail of Broken Dreams\n\n\n\n\...
Chapters for BFfH:
   book chapter_id                                               text
0  BFfH     

This data is pretty clean, because it comes from a a quality digital first source. Front and back matter were already removed in the previous section, but there are many many dates (at the start of each diary entry) and chapter formatting which need to be removed. 

In [9]:
def is_valid_sentence(s):
    if len(s) < 7:
        return False

    bad_endings = ("Mrs.", "Mr.", "Miss.", "Dr.", "“", '"')
    if s.endswith(bad_endings):
        return False

    return True

In [10]:
# Creating entry and sentence DataFrames

base_output_folder = "dataframes/untagged"
os.makedirs(base_output_folder, exist_ok=True)

entry_output_folder = os.path.join(base_output_folder, "all_books_entry_dfs")
sentence_output_folder = os.path.join(base_output_folder, "all_books_sent_dfs")

os.makedirs(entry_output_folder, exist_ok=True)
os.makedirs(sentence_output_folder, exist_ok=True)



for book_name, book_info in books.items():
    with open(book_info["file"], "r", encoding="utf-8") as f:
        raw_text = f.read()

    start = raw_text.find(book_info["start_marker"])
    end = raw_text.find(book_info["end_marker"])
    diary_text = raw_text[start:end]

    sections = re.split(book_info["chapter_pattern"], diary_text)

    all_entries = []
    all_sentences = []

    for i in range(1, len(sections), 2):
        chapter_id = sections[i]
        text = sections[i+1]

        # ---------------------------------------------------------
        # 1. Remove chapter header
        # ---------------------------------------------------------
        text = re.sub(
            r"^Dear Canada:.*?\n+",
            "",
            text,
            flags=re.DOTALL
        ).strip()

        # ---------------------------------------------------------
        # 2. Extract dated entries
        # ---------------------------------------------------------
        matches = list(re.finditer(book_info["date_pattern"], text, re.VERBOSE | re.IGNORECASE))

        for j, match in enumerate(matches):

            date = match.group()
            start = match.end()
            end = matches[j + 1].start() if j + 1 < len(matches) else len(text)

            entry_text = text[start:end]

            # -----------------------------------------------------
            # 3. Clean entry text
            # -----------------------------------------------------
            entry_text = re.sub(r"\n\s*\n", " ", entry_text)
            entry_text = re.sub(r"\s+", " ", entry_text).strip()

            entry_id = f"ch{chapter_id}_e{j+1}"

            all_entries.append({
                "book": book_name,
                "chapter_id": chapter_id,
                "entry_id": entry_id,
                "date": date,
                "entry_text": entry_text
            })

            # -----------------------------------------------------
            # 4. Sentence tokenize
            # -----------------------------------------------------
            sentences = sent_tokenize(entry_text)

            for sentence in sentences:

                sentence = sentence.strip()

                if is_valid_sentence(sentence):
                    all_sentences.append({
                        "book_name": book_name,
                        "chapter_id": chapter_id,
                        "entry_id": entry_id,
                        "date": date,
                        "sentence": sentence
                    })

    entries_df = pd.DataFrame(all_entries)
    sentences_df = pd.DataFrame(all_sentences)

    entries_df.to_csv(os.path.join(entry_output_folder, f"{book_name}_entries.csv"), index=False)
    sentences_df.to_csv(os.path.join(sentence_output_folder, f"{book_name}_sentences.csv"), index=False)

print(f"Diary entry and sentence DataFrames saved to folder: {entry_output_folder} and {sentence_output_folder}")

Diary entry and sentence DataFrames saved to folder: dataframes/untagged/all_books_entry_dfs and dataframes/untagged/all_books_sent_dfs
